# 03 · JTAGulator-style SWD pinout detection

Like a JTAGulator, FaultyCat can **brute-force a target's debug pinout**:
it tries every ordered pair of the 8 scanner-header channels (GP0..GP7),
P(8,2)=56 permutations, driving an SWD handshake on each until it gets a
valid DPIDR back. That tells you which pins are **SWCLK** and **SWDIO** —
no datasheet needed.

This is `cat.scanner`, over the scanner-shell CDC. No high voltage is
involved.

## Wiring

Connect the target's candidate debug pads to the **scanner header**:

- Target **SWCLK** → any GP channel (GP0..GP7)
- Target **SWDIO** → any other GP channel
- Common **GND** between target and FaultyCat

You don't need to know which pad is which — that's what the scan finds.

In [ ]:
import faultycat as fc
SIM = False                 # True = simulator (returns a canned GP2/GP3 match)
cat = fc.connect(simulator=SIM)
cat.scanner                 # shows scanner capabilities

## 1 · Scan for SWD

`swd()` sweeps the 56 permutations; `on_progress` streams the firmware's
progress lines. `NO_MATCH` means nothing valid was found — check wiring,
target power, or that the pins are within GP0..GP7.

In [ ]:
res = cat.scanner.swd(timeout_s=45, on_progress=print)
res                          # SwdScanResult: matched / swclk_gp / swdio_gp

In [ ]:
if res.matched:
    print(f'Found SWD:  SWCLK = GP{res.swclk_gp}   SWDIO = GP{res.swdio_gp}')
    print('Raw firmware lines (DPIDR / targetsel):')
    for line in res.lines:
        print('   ', line)
else:
    print('No SWD match. Nothing wired to the scanner header, wrong pins, or target unpowered.')

## 2 · (Bonus) I2C bus discovery

The same header can brute-force an I2C bus — find SDA/SCL and list the
device addresses that ACK.

In [ ]:
try:
    i2c = cat.scanner.i2c(timeout_s=30)
    if i2c.matched:
        print(f'I2C on SDA=GP{i2c.sda_gp} SCL=GP{i2c.scl_gp} — addresses: {i2c.addresses_hex}')
    else:
        print('No I2C devices found.')
except NotImplementedError as e:
    print('I2C scan needs the full faultycmd:', e)

In [ ]:
cat.close()